# Load-balancer orchestrator: fan out 5,000 calls across 5 spokes

```
            this notebook (orchestrator)
                       |
        notebookutils.notebook.runMultiple([...], concurrency=5)
        /       /       |       \        \
   Spoke A  Spoke B  Spoke C  Spoke D  Spoke E      (5 sub-sessions in parallel)
   1000 x   1000 x   1000 x   1000 x   1000 x
       \       \       |       /        /
        \       \      |      /        /
                 Azure Function (HTTP)             (5,000 total HTTPS calls)
```

Each spoke fires `calls_per_spoke` concurrent HTTPS requests against the function using a `ThreadPoolExecutor` and returns aggregated latency stats. This orchestrator collects all 5 envelopes and prints a combined report.

In [ ]:
# === Configure ===
import json

function_url     = "https://my-fn-app.azurewebsites.net/api/HelloWorld"
function_key     = "REPLACE_WITH_FUNCTION_KEY"
calls_per_spoke  = 1000           # 5 spokes * 1000 = 5000 total
concurrency      = 50             # in-spoke parallelism (threads per spoke)
request_timeout  = 30             # per-request timeout (seconds)
spoke_timeout_s  = 1800           # per-notebook timeout (seconds)

# IMPORTANT: Fabric notebook.run / runMultiple only accept FLAT primitive
# values in `arguments`. Nested dicts/lists silently kill the spawned Spark
# session. Pass complex payloads as a JSON string and parse in the child.
payload_json = json.dumps({"name": "loadtest",
                            "metadata": {"source": "loadbalancer"}})

spokes = ["04-loadgen-spoke-A", "04-loadgen-spoke-B", "04-loadgen-spoke-C",
          "04-loadgen-spoke-D", "04-loadgen-spoke-E"]

total_calls = calls_per_spoke * len(spokes)
print(f"Spokes          : {len(spokes)}")
print(f"Calls per spoke : {calls_per_spoke}")
print(f"Total calls     : {total_calls}")
print(f"In-spoke threads: {concurrency}")

In [ ]:
# === Launch all 5 spokes in parallel ===
import json, time
try:
    from notebookutils import notebook
except Exception:
    from mssparkutils import notebook

# All values FLAT primitives (string/int) -- Fabric runMultiple kills the
# session if any arg is a dict/list.
shared_args = {
    "function_url":    function_url,
    "function_key":    function_key,
    "calls":           calls_per_spoke,
    "concurrency":     concurrency,
    "request_timeout": request_timeout,
    "payload_json":    payload_json,
}

# Fabric runMultiple expects a DAG: {activities:[...], concurrency, timeoutInSeconds}
# Each activity needs a UNIQUE `name` field.
activities = [
    {"name":  f"spoke_{s[-1]}",
     "path":  s,
     "args":  {**shared_args, "shard_id": s[-1]},   # 'A'..'E'
     "timeoutPerCellInSeconds": spoke_timeout_s}
    for s in spokes
]

dag = {
    "activities":       activities,
    "concurrency":      len(spokes),
    "timeoutInSeconds": spoke_timeout_s + 60,
}

t0 = time.time()
results_raw = notebook.runMultiple(dag)
wall_s = time.time() - t0

print(f"Wall time (5 spokes in parallel): {wall_s:.1f}s")
print(f"runMultiple returned type: {type(results_raw).__name__}")
print(f"raw payload preview: {str(results_raw)[:300]}")

In [ ]:
# === Aggregate across spokes ===
import json
import statistics as stats

# runMultiple's return shape varies by runtime. Normalize.
# Each spoke's notebook.exit() value is a JSON string.
def _extract_spoke_results(raw):
    out = {}
    if isinstance(raw, dict):
        # New API: {activityName: {exitVal: "...", exception: None}}
        # Older shape: {notebookPath: {exitValue: "..."}}
        for k, v in raw.items():
            ev = None
            if isinstance(v, dict):
                ev = v.get("exitVal") or v.get("exitValue")
            else:
                ev = v
            if ev:
                try: out[k] = json.loads(ev)
                except Exception: out[k] = {"raw": ev}
    elif isinstance(raw, list):
        for item in raw:
            path = item.get("notebookPath") or item.get("path") or f"spoke_{len(out)}"
            ev   = item.get("exitValue") or item.get("result")
            if ev:
                try: out[path] = json.loads(ev)
                except Exception: out[path] = {"raw": ev}
    return out

spoke_results = _extract_spoke_results(results_raw)
assert spoke_results, f"No spoke results extracted. Raw: {results_raw!r}"

print(f"Got results from {len(spoke_results)} spokes:\n")
print(f"{'spoke':<32} {'ok':>6} {'fail':>6} {'p50':>7} {'p95':>7} {'p99':>7} {'rps':>7}")
print("-" * 80)

all_lat = []
total_ok = total_fail = 0
for name, r in sorted(spoke_results.items()):
    ok   = r.get("count_ok", 0)
    fail = r.get("count_fail", 0)
    p50  = r.get("p50_ms", 0)
    p95  = r.get("p95_ms", 0)
    p99  = r.get("p99_ms", 0)
    rps  = r.get("throughput_rps", 0)
    print(f"{name[-32:]:<32} {ok:>6} {fail:>6} {p50:>7.0f} {p95:>7.0f} {p99:>7.0f} {rps:>7.1f}")
    total_ok   += ok
    total_fail += fail
    all_lat.extend(r.get("latencies_ms", []))

print("-" * 80)
combined_p50 = stats.median(all_lat) if all_lat else 0
combined_p95 = stats.quantiles(all_lat, n=20)[18] if len(all_lat) >= 20 else 0
combined_p99 = stats.quantiles(all_lat, n=100)[98] if len(all_lat) >= 100 else 0
combined_rps = (total_ok + total_fail) / wall_s if wall_s else 0
print(f"{'COMBINED':<32} {total_ok:>6} {total_fail:>6} "
      f"{combined_p50:>7.0f} {combined_p95:>7.0f} {combined_p99:>7.0f} "
      f"{combined_rps:>7.1f}")
print()
print(f"Total wall time : {wall_s:.1f}s")
print(f"Success rate    : {total_ok / (total_ok + total_fail) * 100:.2f}%")